# Hosty 2026: experimental trajectories and a preset OU reference

### Settings and parameters

Calibration: **3 min/frame** and **810.44/512 µm/pixel**.
Set the parameter grid, filtering, simulation count and duration below.
The current movies have 101 frames, so 100 simulation steps cover their full duration.

In [ ]:
from pathlib import Path
import itertools
import json
import sys

import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
import numpy as np
import pandas as pd
import scienceplots
plt.style.use(["science"])
from IPython.display import display
from tqdm.auto import tqdm

MODEL_DIR = next(
    p for ancestor in (Path.cwd(), *Path.cwd().parents)
    for p in (ancestor, ancestor / "paper/section_1/OU-comparison")
    if (p / "OU_model.py").is_file() and (p / "experimental_data/Hosty_2026").is_dir()
)
if str(MODEL_DIR) not in sys.path:
    sys.path.insert(0, str(MODEL_DIR))
import OU_model as ou
import hosty_data


BLUE, ORANGE = "#0072B2", "#D55E00"
conditions = ["NT", "Bispecific", "Rtx"]
cell_types = ["NK", "Tumour"]
groups = [(condition, cell_type) for cell_type in cell_types for condition in conditions]
DATA_DIR = MODEL_DIR / "experimental_data/Hosty_2026"
OUTPUT = MODEL_DIR / "outputs/hosty_2026_bands"
OUTPUT.mkdir(parents=True, exist_ok=True)

hosty_data.self_check()
ou.self_check()

In [ ]:
dt = 3.0
pixel_size_um = 810.44 / 512
min_valid_steps = 20
max_lag = 8
seed = 2026
n_sweep_tracks = 500
n_simulation_steps = 100  # 101 frames spanning 300 min, matching the movie duration

alphas = np.geomspace(0.1, 10.0, 7)   # min^-1: persistence times of 1–10 min
sigma_mus = np.linspace(0.1, 1.2, 7) # µm / min^(3/2)
sigma_rs = np.linspace(0.1, 2.0, 7)  # µm / sqrt(min)


hdi_mass = 0.98  # Interval across OU parameter-setting means
band_quantiles = [0.01, 0.99]  # Pointwise bands for MSD and turning-angle curves
curve_names = ["msd", "turning_cdf"]
for values, positive in [(alphas, True), (sigma_mus, False), (sigma_rs, False)]:
    assert values.ndim == 1 and len(values) and np.isfinite(values).all()
    assert (values > 0).all() if positive else (values >= 0).all()
    assert len(np.unique(values)) == len(values)
assert np.isfinite([dt, pixel_size_um]).all() and dt > 0 and pixel_size_um > 0
assert all(isinstance(v, int) and v > 0 for v in [min_valid_steps, max_lag, n_sweep_tracks, n_simulation_steps])
assert isinstance(seed, int) and seed >= 0 and max_lag >= 3
parameters = [p for p in itertools.product(alphas, sigma_mus, sigma_rs) if p[1] > 0 or p[2] > 0]
assert parameters
assert n_simulation_steps >= max(min_valid_steps, max_lag + 2)
lag_minutes = dt * np.arange(1, max_lag + 1)

print(f"alphas: {alphas}\nsigma_mus: {sigma_mus}\nsigma_rs: {sigma_rs}")
print(f"{len(parameters):,} parameter combinations × {n_sweep_tracks} tracks = "
      f"{len(parameters) * n_sweep_tracks:,} simulations in one shared sweep")
print(f"Stored OU trajectories: {len(parameters) * n_sweep_tracks * (n_simulation_steps + 1) * 2 * 8 / 2**30:.2f} GiB")
assert 0 < hdi_mass <= 1
assert 0 <= band_quantiles[0] < .5 < band_quantiles[1] <= 1

### OU simulations

In [ ]:
simulated_paths = np.empty((len(parameters), n_sweep_tracks, n_simulation_steps + 1, 2))
for parameter_id, (alpha, sigma_mu, sigma_r) in tqdm(enumerate(parameters)):
    simulated_paths[parameter_id] = ou.simulate_ou(
        alpha, sigma_mu, sigma_r, n_sweep_tracks, n_simulation_steps, dt,
        np.random.default_rng([seed, parameter_id, 200]), initial="stationary")

print(f"Generated {len(parameters) * n_sweep_tracks:,} OU trajectories, "
      f"each lasting {n_simulation_steps * dt:g} min")

### Experimental data: import and visualisation

Dummy detections and absent frames remain `NaN`; their time slots are preserved.
Retained tracks need at least `min_valid_steps` observed adjacent-frame steps
and a span of at least `max_lag + 2` frame intervals.

- **Speed:** displacement between consecutive observed frames divided by `dt`;
  use the mean speed for each track.
- **MSD:** average squared displacement at each fixed lag. Both endpoints must
  be observed; intermediate frames may be missing, provided the cell identity is correct.
- **Turning angles:** use three consecutive observed positions and two nonzero
  displacement vectors. Small movements remain sensitive to position uncertainty.

All valid sections contribute to these statistics, including sections after gaps.
The same summary function is used for the data and simulations.

In [ ]:
positions, track_metadata, import_audit = hosty_data.load_hosty(
    DATA_DIR, min_valid_steps=min_valid_steps, min_span_frames=max_lag + 2,
    pixel_size=pixel_size_um,
)
assert len(positions) == len(track_metadata) > 0
assert positions.shape[1] <= simulated_paths.shape[2], "Increase n_simulation_steps and rerun the OU sweep"
scalar_parts, pair_parts, turn_parts = [], [], []
curve_parts = {name: [] for name in curve_names}
for start in range(0, len(positions), 512):
    batch = positions[start:start + 512]
    scalars, curves = ou.summarize_tracks(batch, dt, max_lag)
    scalar_parts.append(
        pd.DataFrame({name: scalars[name] for name in
        ["mean_speed", "observed_duration", "missing_fraction"]})
        )
    for name in curve_names:
        curve_parts[name].append(curves[name])
    valid = np.isfinite(batch).all(axis=2)
    pair_parts.append(np.column_stack([
        (valid[:, lag:] & valid[:, :-lag]).sum(axis=1)
        for lag in range(1, max_lag + 1)
    ]))
    step_squared = np.sum(np.diff(batch, axis=1) ** 2, axis=2)
    moving = np.isfinite(step_squared) & (step_squared > 0)
    turn_parts.append((moving[:, :-1] & moving[:, 1:]).sum(axis=1))

observed = pd.concat([track_metadata, pd.concat(scalar_parts, ignore_index=True)], axis=1)
observed["n_valid_turns"] = np.concatenate(turn_parts)
observed_curves = {name: np.concatenate(parts) for name, parts in curve_parts.items()}
observed_msd_pairs = np.concatenate(pair_parts)
np.testing.assert_array_equal(observed_msd_pairs[:, 0], observed.n_valid_steps)
np.testing.assert_array_equal(np.isfinite(observed_curves["msd"]), observed_msd_pairs > 0)
group_indices = observed.groupby(["condition", "cell_type"], sort=False).indices
assert set(group_indices) == set(groups)
display(observed.groupby(["condition", "cell_type"]).agg(
    tracks=("track_id", "size"), movies=("movie", "nunique"), donors=("donor", "nunique"),
    median_valid_steps=("n_valid_steps", "median"), median_valid_turns=("n_valid_turns", "median"),
    median_missing_fraction=("missing_fraction", "median")))
unavailable = [
    str(p.relative_to(DATA_DIR)) for p in sorted(DATA_DIR.glob("*/*"))
    if p.is_dir() and not list(p.glob("*_trajectories.csv"))]
print("Folders without trajectory exports:", unavailable or "none")

In [ ]:
fig, axes = plt.subplots(
    2, 3, figsize=(15, 9), sharex=True, sharey=True,
    layout="constrained", dpi=300)
for row, cell_type in enumerate(cell_types):
    for col, condition in enumerate(conditions):
        ax = axes[row, col]
        indices = group_indices[(condition, cell_type)]
        plot_paths = []
        for track in positions[indices]:
            missing = np.flatnonzero(~np.isfinite(track).all(axis=1))
            stop = missing[0] if missing.size else len(track)
            if stop >= 2:
                plot_paths.append(track[:stop])
        ax.add_collection(
            LineCollection(plot_paths, colors=[BLUE, ORANGE][row],
            linewidths=.8, alpha=.15, rasterized=True))
        ax.autoscale_view()
        ax.set_aspect("equal", adjustable="box")
        ax.scatter(0, 0, color="black", s=12, zorder=3)
        ax.text(.03, .97, f"{len(plot_paths):,} initial segments\n{len(indices):,} retained tracks",
                transform=ax.transAxes, va="top")
        if row == 0:
            ax.set_title(condition)
        elif row == 1:
            ax.set_xlabel("x displacement (µm)")
        if col == 0:
            ax.set_ylabel(f"{cell_type}\ny displacement (µm)")
# fig.savefig(OUTPUT / "initial_trajectories.png", dpi=300)
plt.show()
plt.close(fig)

### Comparison

<!-- Use every frame of each complete OU trajectory. Experimental step speeds use
only pairs of observed positions in consecutive frames; gaps are never bridged.

For the mean-speed comparison, average the track means within each OU parameter
setting. The blue HDI is the shortest contiguous interval containing at least
`hdi_mass` of these equally weighted parameter means. Each experimental dot is
the mean of the track means in that case, with equal weight per track.

For the step-speed comparison, pool valid steps within each experimental case.
The blue line and band show the mean and standard deviation of all OU steps
pooled across the grid. These include variation within and between parameter
settings. Longer experimental tracks contribute more steps to this second plot.

Speeds are displacements over the 3-minute frame interval. All standard deviations
use `ddof=0`. The intervals describe the preset simulation grid and the spread
of recorded speeds; they do not estimate uncertainty in the experimental means.
MSD and turning-angle comparisons retain their existing pointwise percentile bands. -->

In [ ]:
band_label = f"OU {100 * band_quantiles[0]:g}–{100 * band_quantiles[1]:g} percentile range"
model_speed_rows = []
curve_batches = {name: [] for name in curve_names}

for parameter_id in tqdm(range(len(parameters))):
    paths = simulated_paths[parameter_id]
    scalars, curves = ou.summarize_tracks(paths, dt, max_lag)
    step_speeds = np.linalg.norm(np.diff(paths, axis=1), axis=2) / dt
    step_speeds = step_speeds[np.isfinite(step_speeds)]
    alpha, sigma_mu, sigma_r = parameters[parameter_id]
    model_speed_rows.append(dict(
        alpha=alpha, sigma_mu=sigma_mu, sigma_r=sigma_r,
        mean_track_speed=np.mean(scalars["mean_speed"]), n_tracks=len(paths),
        mean_step_speed=step_speeds.mean(), step_variance=step_speeds.var(ddof=0),
        n_step_samples=len(step_speeds)))
    for name in curve_names:
        curve_batches[name].append(curves[name].astype(np.float32))

model_speed_summary = pd.DataFrame(model_speed_rows)
model_speed_summary["step_std"] = np.sqrt(model_speed_summary.step_variance)
np.testing.assert_allclose(
    model_speed_summary.mean_track_speed,
    model_speed_summary.mean_step_speed)
assert (model_speed_summary.n_step_samples == n_sweep_tracks * n_simulation_steps).all()

# Include both within-setting and between-setting variation in the pooled SD.
weights = model_speed_summary.n_step_samples.to_numpy()
means = model_speed_summary.mean_step_speed.to_numpy()
variances = model_speed_summary.step_variance.to_numpy()
model_step_mean = np.average(means, weights=weights)
model_step_std = np.sqrt(np.average(
    variances + (means - model_step_mean)**2, weights=weights))

observed_speed_rows = []
for condition, cell_type in groups:
    indices = group_indices[(condition, cell_type)]
    step_speeds = np.linalg.norm(np.diff(positions[indices], axis=1), axis=2) / dt
    step_speeds = step_speeds[np.isfinite(step_speeds)]
    assert len(step_speeds) == observed.iloc[indices].n_valid_steps.sum()
    observed_speed_rows.append(dict(
        condition=condition, cell_type=cell_type,
        mean_track_speed=observed.iloc[indices].mean_speed.mean(), n_tracks=len(indices),
        mean_step_speed=step_speeds.mean(), step_std=step_speeds.std(ddof=0),
        n_step_samples=len(step_speeds)))
observed_speed_summary = pd.DataFrame(observed_speed_rows)

model_curve_bands, model_curve_counts = {}, {}
for name, parts in curve_batches.items():
    pooled_curve = np.concatenate(parts)
    model_curve_bands[name] = np.nanquantile(
        pooled_curve, [band_quantiles[0], .5, band_quantiles[1]], axis=0)
    model_curve_counts[name] = np.isfinite(pooled_curve).sum(axis=0)

model_speed_summary.to_csv(OUTPUT / "ou_speed_by_parameter.csv", index=False)
observed_speed_summary.to_csv(OUTPUT / "observed_speed_summary.csv", index=False)
display(observed_speed_summary)
print(
    f"Shared OU reference: {len(parameters):,} parameter settings; "
    f"{model_speed_summary.n_step_samples.sum():,} complete 3-minute steps")
del pooled_curve, curve_batches, parts, step_speeds, model_speed_rows, observed_speed_rows

### Mean track speed: OU HDI and experimental means

Blue shading shows the shortest contiguous interval containing at least 98% of
the parameter-setting mean speeds (or the mass chosen in `hdi_mass`). The blue
line is their average. Orange dots show experimental group means, with each
track weighted equally. The speed axis is logarithmic above 0.001 µm/min.

This empirical HDI is calculated in linear speed units. For a multimodal grid
mixture, a single interval may span regions with few parameter-setting means.
It is a descriptive interval across the grid, not a posterior credible interval.
The [HDI definition](https://arviz-devs.github.io/ArviZ.jl/dev/api/stats/) motivates
using the shortest interval rather than fixed lower and upper percentiles.

In [ ]:
parameter_mean_speeds = model_speed_summary.mean_track_speed.to_numpy()
values = np.sort(parameter_mean_speeds[np.isfinite(parameter_mean_speeds)])
assert len(values) and 0 < hdi_mass <= 1
n_inside = int(np.ceil(hdi_mass * len(values)))
start = np.argmin(values[n_inside - 1:] - values[:len(values) - n_inside + 1])
hdi_low, hdi_high = values[[start, start + n_inside - 1]]
assert np.count_nonzero((values >= hdi_low) & (values <= hdi_high)) >= n_inside
model_mean_track_speed = values.mean()

case_labels = [f"{condition} · {cell_type}" for condition, cell_type in groups]
case_y = np.arange(len(groups))
fig, ax = plt.subplots(figsize=(7.4, 4.2), layout="constrained")
ax.axvspan(
      hdi_low, hdi_high, color=BLUE, alpha=.13,
      label=rf"OU ${100 * hdi_mass:g}\%$ HDI across parameter means")
ax.axvline(
      model_mean_track_speed, color=BLUE, lw=1.6,
      label="OU mean across parameter settings")
ax.scatter(
      observed_speed_summary.mean_track_speed, case_y,
      color=ORANGE, s=35, zorder=3, label="Experimental mean across tracks")
ax.set_yticks(case_y, case_labels)
ax.invert_yaxis()
ax.set_xscale("symlog", linthresh=.001)
ax.xaxis.get_major_locator().set_params(numticks=5)
ax.set_xlim(
    left=max(0, min(hdi_low, observed_speed_summary.mean_track_speed.min()) * .7),
    right=max(hdi_high, observed_speed_summary.mean_track_speed.max(), model_mean_track_speed) * 1.2)
ax.set_xlabel("Mean track speed (µm/min)")
ax.set_title("Mean track speed")
ax.spines[["top", "right"]].set_visible(False)
ax.tick_params(which="both", top=False, right=False)
ax.grid(axis="x", color="0.9", lw=.5)
ax.set_axisbelow(True)
ax.legend(loc="upper center", bbox_to_anchor=(.5, -0.17), frameon=False, fontsize=9)
fig.savefig(OUTPUT / "mean_speed_hdi.png", dpi=300)
fig.savefig(OUTPUT / "mean_speed_hdi.pdf")
plt.show()
plt.close(fig)
print(f"OU {100 * hdi_mass:g}% HDI across parameter means: "
      f"{hdi_low:.4g}–{hdi_high:.4g} µm/min")

### MSD and turning angles

Each curve is first calculated within each track. Blue shows the pointwise
median and central range across simulated tracks; orange shows the pointwise
median across experimental tracks. Thus the turning-angle plot is the median
within-track cumulative distribution, rather than a distribution pooled over all turns.

MSD uses the fixed lags in the settings cell. The table below reports the
minimum number of observed displacement pairs per track at each lag.
Counts of contributing tracks and samples are also saved with the curves.

In [ ]:
condition_colors = dict(zip(conditions, [ORANGE, "#009E73", "#CC79A7"]))
cell_styles = {"NK": "-", "Tumour": "--"}

curve_rows = []

for name in curve_names:
    x = lag_minutes if name == "msd" else ou.TURNING_ANGLE_GRID
    low, median, high = model_curve_bands[name]

    fig, ax = plt.subplots(figsize=(8, 4.8), layout="constrained")

    # Draw the shared OU reference once.
    ax.fill_between(
        x, low, high,
        color=BLUE, alpha=.16, label=band_label,
    )
    ax.plot(
        x, median,
        color=BLUE, lw=2, label="OU median",
    )

    # Overlay all six experimental curves on the same axes.
    for group in groups:
        indices = group_indices[group]
        values = observed_curves[name][indices]
        data = np.nanmedian(values, axis=0)

        ax.plot(
            x, data,
            color=condition_colors[group[0]],
            linestyle=cell_styles[group[1]],
            lw=1.5,
            label=f"{group[0]} · {group[1]}",
        )

        if name == "msd":
            counts = observed_msd_pairs[indices]
        else:
            counts = np.repeat(
                observed.iloc[indices].n_valid_turns.to_numpy()[:, None],
                len(x),
                axis=1,
            )

        for k, coordinate in enumerate(x):
            curve_rows.append(dict(
                condition=group[0],
                cell_type=group[1],
                curve=name,
                lag_or_angle=coordinate,
                ou_low=low[k],
                ou_median=median[k],
                ou_high=high[k],
                observed_median=data[k],
                ou_n_tracks=model_curve_counts[name][k],
                observed_n_tracks=np.isfinite(values[:, k]).sum(),
                observed_n_samples=counts[:, k].sum(),
                observed_min_samples_per_track=counts[:, k].min(),
            ))

    if name == "msd":
        ax.set(
            xlabel="Lag (min)",
            ylabel="Time-averaged MSD (µm²)",
            yscale="log",
            xticks=x,
        )
    else:
        ax.set(
            xlabel="Turning angle (°)",
            ylabel="Within-track cumulative fraction",
            xlim=(0, 180),
            ylim=(0, 1),
        )

    ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(which="both", top=False, right=False)
    ax.legend(
        loc="upper left",
        bbox_to_anchor=(1.02, 1),
        frameon=False,
        fontsize=8,
    )

    fig.savefig(OUTPUT / f"curve_{name}.png", dpi=300)
    fig.savefig(OUTPUT / f"curve_{name}.pdf")
    plt.show()
    plt.close(fig)

curve_table = pd.DataFrame(curve_rows)

display(
    curve_table.query("curve == 'msd'")
    .pivot(
        index=["condition", "cell_type"],
        columns="lag_or_angle",
        values="observed_min_samples_per_track",
    )
    .rename_axis(columns="Lag (min): minimum observed pairs per track")
)